In [7]:
import os
from collections import Counter, defaultdict

import pandas as pd
from graphviz import Digraph

# =========================================
# CONFIG: Local-window drift around milestones (not broad phases)
# =========================================
INPUT_FILE = "../data/bpi-challenge-2017/bpi_2017_cleaned.csv"
OUTPUT_DIR = "../results/phase_drift_analysis_local"
os.makedirs(OUTPUT_DIR, exist_ok=True)

APP_TYPES = ["New credit", "Limit raise"]
MILESTONES = [19, 24, 25, 49]  # cac tuan ban le
WINDOW_SIZE_WEEKS = 1            # so sanh sat moc: [m-1] vs [m]

# Chong spaghetti
VARIANT_COVERAGE = 0.65          # giu cac variant pho bien toi 65% coverage
MIN_EDGE_FREQ = 15               # bo canh qua hiem
TOP_OUTGOING_PER_NODE = 2        # moi node chi giu toi da 2 canh ra manh nhat
# =========================================
# 1. LOAD & PREPARE
# =========================================
df = pd.read_csv(INPUT_FILE)
df = df[df["lifecycle:transition"] == "complete"].copy()
df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], errors="coerce")
df = df.dropna(subset=["time:timestamp"])
df["week"] = df["time:timestamp"].dt.isocalendar().week.astype(int)
df = df.sort_values(["case:concept:name", "time:timestamp"]).copy()

# =========================================
# 2. HELPERS
# =========================================
def case_variant_table(df_sub):
    seq = (
        df_sub.groupby("case:concept:name")["concept:name"]
        .apply(tuple)
        .reset_index(name="variant")
    )
    vc = seq["variant"].value_counts().reset_index()
    vc.columns = ["variant", "count"]
    vc["ratio"] = vc["count"] / vc["count"].sum()
    vc["cum_ratio"] = vc["ratio"].cumsum()
    return seq, vc


def filter_by_top_variants(df_sub, coverage=0.8):
    if df_sub.empty:
        return df_sub.copy(), pd.DataFrame(columns=["variant", "count", "ratio", "cum_ratio"]), 0

    seq, vc = case_variant_table(df_sub)
    keep_variants = set(vc[vc["cum_ratio"] <= coverage]["variant"].tolist())
    if not keep_variants and not vc.empty:
        keep_variants = {vc.iloc[0]["variant"]}

    kept_cases = seq[seq["variant"].isin(keep_variants)]["case:concept:name"]
    filtered = df_sub[df_sub["case:concept:name"].isin(kept_cases)].copy()
    return filtered, vc, len(kept_cases)


def build_edge_stats(df_sub):
    edge_freq = Counter()
    edge_waits = defaultdict(list)

    for _, g in df_sub.groupby("case:concept:name"):
        g = g.sort_values("time:timestamp")
        acts = g["concept:name"].tolist()
        ts = g["time:timestamp"].tolist()

        for i in range(len(acts) - 1):
            e = (acts[i], acts[i + 1])
            edge_freq[e] += 1
            wait_s = (ts[i + 1] - ts[i]).total_seconds()
            if wait_s >= 0:
                edge_waits[e].append(wait_s)

    rows = []
    for e, f in edge_freq.items():
        waits = edge_waits.get(e, [])
        rows.append(
            {
                "src": e[0],
                "tgt": e[1],
                "freq": f,
                "avg_wait_h": (sum(waits) / len(waits) / 3600) if waits else 0.0,
                "p90_wait_h": pd.Series(waits).quantile(0.9) / 3600 if waits else 0.0,
            }
        )
    return pd.DataFrame(rows)


def prune_edges(df_edges, min_freq=8, top_outgoing=2):
    if df_edges.empty:
        return df_edges.copy()

    x = df_edges[df_edges["freq"] >= min_freq].copy()
    if x.empty:
        return x

    x = x.sort_values(["src", "freq", "p90_wait_h"], ascending=[True, False, False])
    x["out_rank"] = x.groupby("src").cumcount() + 1
    x = x[x["out_rank"] <= top_outgoing].drop(columns=["out_rank"])
    return x


def draw_dfg(df_edges, title, file_path):
    dot = Digraph(comment=title)
    dot.attr(rankdir="LR", nodesep="0.4", ranksep="0.8", fontsize="12")

    nodes = sorted(set(df_edges["src"]).union(set(df_edges["tgt"])))
    for n in nodes:
        short_name = n.replace("Application", "App").replace("Accepted", "Acc")
        dot.node(n, short_name, shape="box", style="rounded,filled", fillcolor="#f9f9f9")

    for _, r in df_edges.iterrows():
        label = f"f={int(r['freq'])}\\np90={r['p90_wait_h']:.1f}h"
        penwidth = str(0.8 + r["freq"] / max(1, df_edges["freq"].max()) * 3.2)
        dot.edge(r["src"], r["tgt"], label=label, penwidth=penwidth)

    dot.render(file_path, format="png", cleanup=True)


def local_windows(df_all, milestone, w=2):
    before = df_all[(df_all["week"] >= milestone - w) & (df_all["week"] < milestone)]
    after = df_all[(df_all["week"] >= milestone) & (df_all["week"] < milestone + w)]
    return before, after


# =========================================
# 3. BUILD LOCAL DFG + BOTTLENECK COMPARISON
# =========================================
comparison_rows = []

for m in MILESTONES:
    print(f"\n=== Milestone W{m}: local window +/- {WINDOW_SIZE_WEEKS} tuan ===")
    before_all, after_all = local_windows(df, m, WINDOW_SIZE_WEEKS)

    milestone_dir = os.path.join(OUTPUT_DIR, f"W{m}")
    os.makedirs(milestone_dir, exist_ok=True)

    for app_type in APP_TYPES:
        app_tag = app_type.replace(" ", "_").lower()
        app_dir = os.path.join(milestone_dir, app_tag)
        os.makedirs(app_dir, exist_ok=True)

        before = before_all[before_all["case:ApplicationType"] == app_type].copy()
        after = after_all[after_all["case:ApplicationType"] == app_type].copy()

        # Loc theo top variants de giam nhieu do long-tail behavior
        before_f, before_vc, before_cases = filter_by_top_variants(before, VARIANT_COVERAGE)
        after_f, after_vc, after_cases = filter_by_top_variants(after, VARIANT_COVERAGE)

        before_edges = prune_edges(build_edge_stats(before_f), MIN_EDGE_FREQ, TOP_OUTGOING_PER_NODE)
        after_edges = prune_edges(build_edge_stats(after_f), MIN_EDGE_FREQ, TOP_OUTGOING_PER_NODE)

        if not before_edges.empty:
            draw_dfg(
                before_edges,
                f"W{m} BEFORE ({app_type})",
                os.path.join(app_dir, "before_dfg"),
            )

        if not after_edges.empty:
            draw_dfg(
                after_edges,
                f"W{m} AFTER ({app_type})",
                os.path.join(app_dir, "after_dfg"),
            )

        # So sanh bottleneck edge-level: freq + waiting time
        merged = before_edges.merge(
            after_edges,
            on=["src", "tgt"],
            how="outer",
            suffixes=("_before", "_after"),
        ).fillna(0)

        if not merged.empty:
            merged["delta_freq"] = merged["freq_after"] - merged["freq_before"]
            merged["delta_p90_wait_h"] = merged["p90_wait_h_after"] - merged["p90_wait_h_before"]
            merged["milestone_week"] = m
            merged["application_type"] = app_type
            merged["before_cases_kept"] = before_cases
            merged["after_cases_kept"] = after_cases
            comparison_rows.append(merged)

        # Luu thong ke variant de kiem tra coverage
        before_vc.head(30).to_csv(os.path.join(app_dir, "before_top_variants.csv"), index=False)
        after_vc.head(30).to_csv(os.path.join(app_dir, "after_top_variants.csv"), index=False)

if comparison_rows:
    df_cmp = pd.concat(comparison_rows, ignore_index=True)
    df_cmp = df_cmp.sort_values(
        ["milestone_week", "application_type", "delta_p90_wait_h", "delta_freq"],
        ascending=[True, True, False, False],
    )
    df_cmp.to_csv(os.path.join(OUTPUT_DIR, "bottleneck_local_window_comparison.csv"), index=False)
    print(f"\nSaved comparison table: {os.path.join(OUTPUT_DIR, 'bottleneck_local_window_comparison.csv')}")
else:
    print("\nNo comparable edges after filtering. Consider lowering MIN_EDGE_FREQ or increasing WINDOW_SIZE_WEEKS.")

print(f"\nDone. Output dir: {OUTPUT_DIR}")


=== Milestone W19: local window +/- 1 tuan ===

=== Milestone W24: local window +/- 1 tuan ===

=== Milestone W25: local window +/- 1 tuan ===

=== Milestone W49: local window +/- 1 tuan ===

Saved comparison table: ../results/phase_drift_analysis_local\bottleneck_local_window_comparison.csv

Done. Output dir: ../results/phase_drift_analysis_local


In [2]:
import pandas as pd
import os
from graphviz import Digraph

# =========================================
# CONFIG
# =========================================
INPUT_FILE = "../data/bpi-challenge-2017/bpi_2017_cleaned.csv"
OUTPUT_DIR = "../results/trace_analysis"

TARGET_CASE_NUMS = [19576, 24076]

# =========================================
# 1. LOAD DATA
# =========================================
print("Loading data...")
df = pd.read_csv(INPUT_FILE)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], errors="coerce")
df = df.dropna(subset=["time:timestamp"])

# =========================================
# 2. PREPARE OUTPUT
# =========================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================
# 3. SORT DATA
# =========================================
df = df.sort_values(["case:concept:name", "time:timestamp"])

# =========================================
# 4. GET CASE IDS
# =========================================
case_ids = df["case:concept:name"].drop_duplicates().tolist()

print("Total cases:", len(case_ids))

# Convert index to actual case id
TARGET_CASES = []
for i in TARGET_CASE_NUMS:
    if i - 1 < len(case_ids):
        TARGET_CASES.append(case_ids[i - 1])
    else:
        print("Case index out of range:", i)

print("Target cases:", TARGET_CASES)

# =========================================
# 5. TRACE ANALYSIS (SINGLE CASE)
# =========================================
def analyze_trace(group, case_id):
    print("\n==============================")
    print("TRACE", case_id)
    print("==============================")

    group = group.sort_values("time:timestamp")

    # Flow
    flow = [
        f"{row['concept:name']}({row['lifecycle:transition']})"
        for _, row in group.iterrows()
    ]
    print("\nFlow:")
    print(" -> ".join(flow))

    # Loop detection
    activity_full = group["concept:name"] + "(" + group["lifecycle:transition"] + ")"
    counts = activity_full.value_counts()
    loops = counts[counts > 1]

    print("\nLoop:")
    if len(loops) > 0:
        print(loops)
    else:
        print("No loop")

    # Timestamp check
    if not group["time:timestamp"].is_monotonic_increasing:
        print("Timestamp ERROR")
    else:
        print("Timestamp OK")

    # Business logic
    activities = set(group["concept:name"])

    if "A_Accepted" not in activities:
        print("Missing A_Accepted")

    if "O_Sent" not in activities:
        print("Missing O_Sent")

    if "A_Cancelled" in activities:
        print("Case cancelled")

    if "A_Complete" in activities:
        print("Case completed")


# =========================================
# 6. DRAW TRACE GRAPH
# =========================================
def draw_trace_graph(group, case_id):
    dot = Digraph(comment=f"Trace {case_id}", format="png")

    group = group.sort_values("time:timestamp")
    events = group.reset_index(drop=True)

    for i, row in events.iterrows():
        activity = row["concept:name"]
        lifecycle = row["lifecycle:transition"]
        timestamp = row["time:timestamp"]

        label = f"{activity}\n({lifecycle})\n{timestamp}"

        color = "black"

        if lifecycle == "withdraw":
            color = "red"
        elif lifecycle == "schedule":
            color = "blue"
        elif lifecycle == "complete":
            color = "green"

        if activity == "A_Accepted":
            color = "darkgreen"
        elif activity == "A_Cancelled":
            color = "red"
        elif activity == "O_Sent":
            color = "orange"

        dot.node(str(i), label, color=color)

    for i in range(len(events) - 1):
        dot.edge(str(i), str(i + 1))

    safe_case_id = str(case_id).replace("/", "_")
    output_path = os.path.join(OUTPUT_DIR, safe_case_id)

    dot.render(output_path, view=False)
    print("Saved graph:", output_path)


# =========================================
# 7. BEFORE vs AFTER ANALYSIS
# =========================================
def analyze_before_after(df, case_ids, split_index, label):
    print("\n==============================")
    print("ANALYSIS AROUND", label)
    print("==============================")

    before_cases = case_ids[:split_index]
    after_cases = case_ids[split_index:]

    df_before = df[df["case:concept:name"].isin(before_cases)]
    df_after = df[df["case:concept:name"].isin(after_cases)]

    print("Before cases:", len(before_cases))
    print("After cases:", len(after_cases))

    # Average trace length
    len_before = df_before.groupby("case:concept:name").size().mean()
    len_after = df_after.groupby("case:concept:name").size().mean()

    print("\nAverage trace length:")
    print("Before:", round(len_before, 2))
    print("After :", round(len_after, 2))

    # Activity distribution
    act_before = df_before["concept:name"].value_counts(normalize=True)
    act_after = df_after["concept:name"].value_counts(normalize=True)

    diff = (act_after - act_before).fillna(0).sort_values(ascending=False)

    print("\nTop activity increase:")
    print(diff.head(10))

    print("\nTop activity decrease:")
    print(diff.tail(10))

    # Loop level
    def avg_loop(df_part):
        loops = []
        for _, g in df_part.groupby("case:concept:name"):
            max_loop = g["concept:name"].value_counts().max()
            loops.append(max_loop)
        return sum(loops) / len(loops)

    print("\nAverage loop level:")
    print("Before:", round(avg_loop(df_before), 2))
    print("After :", round(avg_loop(df_after), 2))

    # Duration
    def avg_duration(df_part):
        durations = []
        for _, g in df_part.groupby("case:concept:name"):
            d = g["time:timestamp"].max() - g["time:timestamp"].min()
            durations.append(d.total_seconds())
        return sum(durations) / len(durations)

    print("\nAverage duration (seconds):")
    print("Before:", round(avg_duration(df_before), 2))
    print("After :", round(avg_duration(df_after), 2))

    # Outcome rate
    def outcome_rate(df_part, activity):
        total = df_part["case:concept:name"].nunique()
        has_act = df_part[df_part["concept:name"] == activity]["case:concept:name"].nunique()
        return has_act / total if total > 0 else 0

    print("\nOutcome rate:")
    print("Accepted Before:", round(outcome_rate(df_before, "A_Accepted"), 3))
    print("Accepted After :", round(outcome_rate(df_after, "A_Accepted"), 3))

    print("Cancelled Before:", round(outcome_rate(df_before, "A_Cancelled"), 3))
    print("Cancelled After :", round(outcome_rate(df_after, "A_Cancelled"), 3))


# =========================================
# 8. RUN TRACE ANALYSIS
# =========================================
df_subset = df[df["case:concept:name"].isin(TARGET_CASES)]

for case_id, group in df_subset.groupby("case:concept:name"):
    analyze_trace(group, case_id)
    draw_trace_graph(group, case_id)

# =========================================
# 9. RUN DRIFT ANALYSIS
# =========================================
analyze_before_after(df, case_ids, 19576, "TRACE 19576")
analyze_before_after(df, case_ids, 24076, "TRACE 24076")

print("\nDone.")

Loading data...
Total cases: 31509
Target cases: ['Application_264466585', 'Application_54290162']

TRACE Application_264466585

Flow:
A_Create Application(complete) -> A_Submitted(complete) -> W_Handle leads(schedule) -> W_Handle leads(start) -> W_Handle leads(complete) -> W_Complete application(schedule) -> W_Complete application(start) -> A_Concept(complete) -> W_Complete application(suspend) -> A_Accepted(complete) -> O_Create Offer(complete) -> O_Created(complete) -> O_Create Offer(complete) -> O_Created(complete) -> O_Sent (mail and online)(complete) -> O_Sent (mail and online)(complete) -> W_Complete application(ate_abort) -> W_Call after offers(schedule) -> W_Call after offers(start) -> A_Complete(complete) -> W_Call after offers(suspend) -> W_Call after offers(resume) -> W_Call after offers(suspend) -> W_Call after offers(ate_abort) -> W_Validate application(schedule) -> W_Validate application(start) -> A_Validating(complete) -> W_Validate application(suspend) -> W_Validate ap

In [5]:
# Param sweep nhanh de chon bo loc giam spaghetti
from itertools import product


def edge_count_for_params(var_cov, min_freq, top_out):
    stats = []
    for m in MILESTONES:
        before_all, after_all = local_windows(df, m, WINDOW_SIZE_WEEKS)
        for app_type in APP_TYPES:
            before = before_all[before_all["case:ApplicationType"] == app_type].copy()
            after = after_all[after_all["case:ApplicationType"] == app_type].copy()

            before_f, _, _ = filter_by_top_variants(before, var_cov)
            after_f, _, _ = filter_by_top_variants(after, var_cov)

            b = prune_edges(build_edge_stats(before_f), min_freq, top_out)
            a = prune_edges(build_edge_stats(after_f), min_freq, top_out)

            stats.append(
                {
                    "milestone": m,
                    "app_type": app_type,
                    "before_edges": len(b),
                    "after_edges": len(a),
                    "var_cov": var_cov,
                    "min_freq": min_freq,
                    "top_out": top_out,
                }
            )
    return pd.DataFrame(stats)


candidates = []
for var_cov, min_freq, top_out in product([0.65, 0.70, 0.75, 0.80], [8, 10, 12, 15], [2, 3]):
    t = edge_count_for_params(var_cov, min_freq, top_out)
    candidates.append(
        {
            "var_cov": var_cov,
            "min_freq": min_freq,
            "top_out": top_out,
            "mean_edges": (t["before_edges"].mean() + t["after_edges"].mean()) / 2,
            "min_edges": min(t["before_edges"].min(), t["after_edges"].min()),
            "max_edges": max(t["before_edges"].max(), t["after_edges"].max()),
        }
    )

cand_df = pd.DataFrame(candidates).sort_values(["mean_edges", "max_edges"])
print("Top 10 bo tham so it spaghetti nhat:")
print(cand_df.head(10).to_string(index=False))

print("\nTop 10 bo tham so can bang (mean_edges trong [10, 20], min_edges >= 5):")
balanced = cand_df[(cand_df["mean_edges"].between(10, 20)) & (cand_df["min_edges"] >= 5)]
print(balanced.head(10).to_string(index=False))

Top 10 bo tham so it spaghetti nhat:
 var_cov  min_freq  top_out  mean_edges  min_edges  max_edges
    0.65        15        2     19.9375         12         29
    0.65        12        2     20.6250         12         29
    0.70        15        2     21.0000         12         30
    0.65        10        2     21.0625         12         29
    0.65        15        3     21.0625         12         31
    0.65         8        2     21.6250         12         30
    0.70        12        2     21.6250         13         30
    0.75        15        2     21.7500         12         30
    0.65        12        3     21.9375         12         32
    0.70        10        2     22.0625         14         31

Top 10 bo tham so can bang (mean_edges trong [10, 20], min_edges >= 5):
 var_cov  min_freq  top_out  mean_edges  min_edges  max_edges
    0.65        15        2     19.9375         12         29


In [6]:
# So sanh do on dinh theo kich thuoc cua so gan moc

def edge_count_for_window(window_size, var_cov=0.65, min_freq=15, top_out=2):
    stats = []
    for m in MILESTONES:
        before_all, after_all = local_windows(df, m, window_size)
        for app_type in APP_TYPES:
            before = before_all[before_all["case:ApplicationType"] == app_type].copy()
            after = after_all[after_all["case:ApplicationType"] == app_type].copy()

            before_f, _, before_cases = filter_by_top_variants(before, var_cov)
            after_f, _, after_cases = filter_by_top_variants(after, var_cov)

            b = prune_edges(build_edge_stats(before_f), min_freq, top_out)
            a = prune_edges(build_edge_stats(after_f), min_freq, top_out)

            stats.append(
                {
                    "window": window_size,
                    "milestone": m,
                    "app_type": app_type,
                    "before_cases": before_cases,
                    "after_cases": after_cases,
                    "before_edges": len(b),
                    "after_edges": len(a),
                }
            )
    return pd.DataFrame(stats)

w1 = edge_count_for_window(1)
w2 = edge_count_for_window(2)

print("Window=1 summary")
print(w1[["before_cases", "after_cases", "before_edges", "after_edges"]].describe().to_string())
print("\nWindow=2 summary")
print(w2[["before_cases", "after_cases", "before_edges", "after_edges"]].describe().to_string())

Window=1 summary
       before_cases  after_cases  before_edges  after_edges
count        8.0000     8.000000      8.000000     8.000000
mean       484.0000   518.500000     16.375000    17.125000
std        418.7321   457.487549      5.853875     6.621124
min         88.0000    87.000000     10.000000     9.000000
25%        100.2500    98.500000     11.000000    11.750000
50%        411.0000   456.500000     16.000000    17.500000
75%        891.5000   931.000000     21.500000    23.000000
max        959.0000  1086.000000     23.000000    24.000000

Window=2 summary
       before_cases  after_cases  before_edges  after_edges
count      8.000000     8.000000      8.000000     8.000000
mean     779.750000   825.000000     19.500000    20.375000
std      689.771753   738.789551      7.270292     7.744814
min      140.000000   133.000000     12.000000    12.000000
25%      158.500000   157.500000     12.750000    13.750000
50%      663.000000   683.500000     19.500000    19.500000
75%  

In [8]:
# Tong hop ket qua cuoi cung de doc bottleneck ro rang
cmp_path = "../results/phase_drift_analysis_local/bottleneck_local_window_comparison.csv"
res = pd.read_csv(cmp_path)

print("Rows:", len(res))

# 1) Cac canh co do tre tang manh nhat sau moc (bottleneck nang len)
print("\n=== TOP BOTTLENECK TANG (delta_p90_wait_h > 0) ===")
pos = res.sort_values(["milestone_week", "application_type", "delta_p90_wait_h", "delta_freq"], ascending=[True, True, False, False])
for (m, app), g in pos.groupby(["milestone_week", "application_type"]):
    print(f"\nW{m} | {app}")
    cols = [
        "src", "tgt", "freq_before", "freq_after",
        "p90_wait_h_before", "p90_wait_h_after", "delta_p90_wait_h", "delta_freq"
    ]
    print(g.head(3)[cols].to_string(index=False))

# 2) Cac canh thay doi luong lon nhat (|delta_freq| cao)
print("\n=== TOP FLOW SHIFT (|delta_freq| cao) ===")
for (m, app), g in res.groupby(["milestone_week", "application_type"]):
    h = g.assign(abs_delta_freq=g["delta_freq"].abs())\
         .sort_values(["abs_delta_freq", "delta_p90_wait_h"], ascending=[False, False])\
         .head(3)
    cols = ["src", "tgt", "freq_before", "freq_after", "delta_freq", "delta_p90_wait_h"]
    print(f"\nW{m} | {app}")
    print(h[cols].to_string(index=False))

# 3) Bang tom tat cap milestone/app
summary_rows = []
for (m, app), g in res.groupby(["milestone_week", "application_type"]):
    top_delay = g.sort_values(["delta_p90_wait_h", "delta_freq"], ascending=[False, False]).head(1).iloc[0]
    top_shift = g.assign(abs_delta_freq=g["delta_freq"].abs())\
                 .sort_values(["abs_delta_freq", "delta_p90_wait_h"], ascending=[False, False]).head(1).iloc[0]

    summary_rows.append({
        "milestone_week": m,
        "application_type": app,
        "top_delay_edge": f"{top_delay['src']} -> {top_delay['tgt']}",
        "top_delay_h": top_delay["delta_p90_wait_h"],
        "top_delay_freq_shift": top_delay["delta_freq"],
        "top_shift_edge": f"{top_shift['src']} -> {top_shift['tgt']}",
        "top_shift_freq": top_shift["delta_freq"],
        "top_shift_delay_h": top_shift["delta_p90_wait_h"],
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["milestone_week", "application_type"])
summary_out = "../results/phase_drift_analysis_local/bottleneck_summary_interpretation.csv"
summary_df.to_csv(summary_out, index=False)

print("\n=== SUMMARY TABLE ===")
print(summary_df.to_string(index=False))
print(f"\nSaved summary: {summary_out}")

Rows: 153

=== TOP BOTTLENECK TANG (delta_p90_wait_h > 0) ===

W19 | Limit raise
                     src                    tgt  freq_before  freq_after  p90_wait_h_before  p90_wait_h_after  delta_p90_wait_h  delta_freq
            A_Validating             O_Returned         29.0        37.0           0.011262          0.027681          0.016419         8.0
          O_Create Offer              O_Created         48.0        26.0           0.000440          0.000549          0.000109       -22.0
O_Sent (mail and online) W_Complete application         35.0        16.0           0.000011          0.000018          0.000007       -19.0

W19 | New credit
         src                    tgt  freq_before  freq_after  p90_wait_h_before  p90_wait_h_after  delta_p90_wait_h  delta_freq
  O_Returned W_Validate application          0.0        40.0                0.0         50.877713         50.877713        40.0
  O_Returned               A_Denied          0.0        30.0                0.0      